In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import StandardScaler

# Load Data

In [2]:
data = pd.read_csv('./analysis.csv')

## Preprocessing

In [3]:
# Patients or patient advocates
data['PatScoreArzt'] = data[['PatItem1Arzt', 'PatItem2Arzt', 'PatItem3Arzt','PatItem4Arzt']].sum(axis=1)
data['PatScoreKI'] = data[['PatItem1KI', 'PatItem2KI', 'PatItem3KI','PatItem4KI']].sum(axis=1)

# Expert 1: Sports scientist
data['Exp1ScoreArzt'] = data[['ExpItem1Arzt', 'ExpItem2Arzt', 'ExpItem3Arzt','ExpItem4Arzt']].sum(axis=1)
data['Exp1ScoreKI'] = data[['ExpItem1KI', 'ExpItem2KI', 'ExpItem3KI','ExpItem4KI']].sum(axis=1)

# Expert 2: Geriatric senior physician
data['Exp2ScoreArzt'] = data[['ExpItem1Arzt.1', 'ExpItem2Arzt.1', 'ExpItem3Arzt.1','ExpItem4Arzt.1']].sum(axis=1)
data['Exp2ScoreKI'] = data[['ExpItem1KI.1', 'ExpItem2KI.1', 'ExpItem3KI.1','ExpItem4KI.1']].sum(axis=1)

# Additional Analyses

## Functions

In [4]:
def bootstrap_ci(data1, data2, effect_fn, n_boot=1000, alpha=0.05, random_state=42):
    """
    Generic paired bootstrap CI for an effect size function.
    """
    rng = np.random.default_rng(random_state)
    n = len(data1)
    stats = []

    data1 = np.asarray(data1)
    data2 = np.asarray(data2)

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        stats.append(effect_fn(data1[idx], data2[idx]))

    lower = np.percentile(stats, 100 * (alpha / 2))
    upper = np.percentile(stats, 100 * (1 - alpha / 2))

    return lower, upper
    

def paired_t_test(array1, array2, n_boot=1000):
    """
    Performs a paired t-test to compare the means of two related groups 
    and calculates Cohen's d with a bootstrap confidence interval.

    Args:
        array1 (array-like): First set of paired observations.
        array2 (array-like): Second set of paired observations.
        n_boot (int): Number of bootstrap iterations for the confidence interval.

    Returns:
        dict: Results containing the test type, means, standard deviations, 
              Cohen's d, its CI, the t-statistic, and the p-value.
    """
    stat, p = stats.ttest_rel(array1, array2)

    diff = np.asarray(array1) - np.asarray(array2)
    cohen_d = np.mean(diff) / np.std(diff, ddof=1)

    d_ci = bootstrap_ci(
        array1, array2,
        effect_fn=lambda a, b: np.mean(a - b) / np.std(a - b, ddof=1),
        n_boot=n_boot
    )

    return {
        "test": "paired_t_test",
        "mean_array1": np.mean(array1),
        "mean_array2": np.mean(array2),
        "std_array1": np.std(array1, ddof=1),
        "std_array2": np.std(array2, ddof=1),
        "cohen_d": cohen_d,
        "cohen_d_ci": d_ci,
        "statistic": stat,
        "p_value": p
    }


def independent_t_test(array1, array2, n_boot=1000, random_state=42):
    """
    Independent t-test with Cohen's d calculation and a bootstrap confidence interval.

    Args:
        array1 (array-like): First set of independent observations.
        array2 (array-like): Second set of independent observations.
        n_boot (int): Number of bootstrap iterations for the confidence interval.
        random_state (int): Seed for the random number generator.

    Returns:
        dict: Results containing the test type, means, standard deviations, 
              Cohen's d, its CI, the t-statistic, and the p-value.
    """
    stat, p = stats.ttest_ind(array1, array2)

    array1 = np.asarray(array1)
    array2 = np.asarray(array2)

    n1 = len(array1)
    n2 = len(array2)

    s1 = np.std(array1, ddof=1)
    s2 = np.std(array2, ddof=1)

    pooled_sd = np.sqrt(
        ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) /
        (n1 + n2 - 2)
    )

    cohen_d = (np.mean(array1) - np.mean(array2)) / pooled_sd

    rng = np.random.default_rng(random_state)
    boot_d = []

    for _ in range(n_boot):
        idx1 = rng.integers(0, n1, n1)
        idx2 = rng.integers(0, n2, n2)

        b = array1[idx1]
        a = array2[idx2]

        s1_b = np.std(a, ddof=1)
        s2_b = np.std(b, ddof=1)

        pooled_sd_b = np.sqrt(
            ((len(a) - 1) * s1_b**2 + (len(b) - 1) * s2_b**2) /
            (len(a) + len(b) - 2)
        )

        boot_d.append((np.mean(a) - np.mean(b)) / pooled_sd_b)

    ci_low = np.percentile(boot_d, 2.5)
    ci_high = np.percentile(boot_d, 97.5)

    return {
        "test": "independent_t_test",
        "mean_array1": np.mean(array1),
        "mean_array2": np.mean(array2),
        "std_array1": s1,
        "std_array2": s2,
        "cohen_d": cohen_d,
        "cohen_d_ci": (ci_low, ci_high),
        "statistic": stat,
        "p_value": p
    }


def wilcoxon_test(array1, array2, n_boot=1000):
    """
    Performs a Wilcoxon signed-rank test for paired samples and 
    calculates the rank-biserial correlation with a bootstrap confidence interval.

    Args:
        array1 (array-like): First set of paired observations.
        array2 (array-like): Second set of paired observations.
        n_boot (int): Number of bootstrap iterations for the confidence interval.

    Returns:
        dict: Results containing the test type, medians, Interquartile Ranges (IQR), 
              rank-biserial correlation, its CI, the test statistic, and the p-value.
    """
    stat, p = stats.wilcoxon(
        array1,
        array2,
        zero_method='wilcox',
        mode='auto'
    )

    diff = np.asarray(array1) - np.asarray(array2)
    diff_nz = diff[diff != 0]

    if len(diff_nz) == 0:
        rank_biserial = np.nan
        r_ci = (np.nan, np.nan)
    else:
        ranks = stats.rankdata(np.abs(diff_nz))
        W_pos = np.sum(ranks[diff_nz > 0])
        W_neg = np.sum(ranks[diff_nz < 0])
        rank_biserial = (W_pos - W_neg) / (W_pos + W_neg)

        # bootstrap CI
        r_ci = bootstrap_ci(
            array1, array2,
            effect_fn=lambda a, b: (
                lambda d: (
                    np.sum(stats.rankdata(np.abs(d[d != 0]))[d[d != 0] > 0]) -
                    np.sum(stats.rankdata(np.abs(d[d != 0]))[d[d != 0] < 0])
                ) / (
                    np.sum(stats.rankdata(np.abs(d[d != 0]))[d[d != 0] > 0]) +
                    np.sum(stats.rankdata(np.abs(d[d != 0]))[d[d != 0] < 0])
                ) if len(d[d != 0]) > 0 else np.nan
            )(a - b),
            n_boot=n_boot
        )

    return {
        "test": "wilcoxon_signed_rank",
        "median_array1": np.median(array1),
        "median_array2": np.median(array2),
        "iqr_array1": tuple(np.percentile(array1, [25, 75])),
        "iqr_array2": tuple(np.percentile(array2, [25, 75])),
        "rank_biserial": rank_biserial,
        "rank_biserial_ci": r_ci,
        "statistic": stat,
        "p_value": p
    }


def rank_biserial_fn(a, b):
    """Calculates the rank-biserial correlation for two independent samples using the Mann-Whitney U statistic."""
    s, _ = stats.mannwhitneyu(a, b, alternative='two-sided')
    return (2 * s) / (len(a) * len(b)) - 1
    

def mann_whitney_u_test(array1, array2, n_boot=1000):
    """
    Perform a Mann-Whitney U test for two independent samples and calculate 
    the rank-biserial correlation with a bootstrap confidence interval.

    Args:
        array1 (array-like): First set of observations.
        array2 (array-like): Second set of observations.
        n_boot (int): Number of bootstrap iterations for the confidence interval.

    Returns:
        dict: Results containing the test type, medians, Interquartile Ranges (IQR), 
              rank-biserial correlation, its CI, the test statistic, and the p-value.
    """
    stat, p = stats.mannwhitneyu(
        array1,
        array2,
        alternative='two-sided'
    )

    n1 = len(array1)
    n2 = len(array2)

    rank_biserial = (2 * stat) / (n1 * n2) - 1

    rng = np.random.default_rng()
    boot_stats = []
    for _ in range(n_boot):
        idx1 = rng.integers(0, n1, n1)
        idx2 = rng.integers(0, n2, n2)
        boot_stats.append(rank_biserial_fn(np.asarray(array1)[idx1], np.asarray(array2)[idx2]))

    ci_low = np.percentile(boot_stats, 2.5)
    ci_high = np.percentile(boot_stats, 97.5)

    return {
        "test": "mann_whitney_u",
        "median_array1": np.median(array1),
        "median_array2": np.median(array2),
        "iqr_array1": tuple(np.percentile(array1, [25, 75])),
        "iqr_array2": tuple(np.percentile(array2, [25, 75])),
        "rank_biserial": rank_biserial,
        "rank_biserial_ci": (ci_low, ci_high),
        "statistic": stat,
        "p_value": p
    }


def spearman_rho(x, y, n_boot=1000, alpha=0.05, random_state=42):
    """
    Calculates the Spearman rank correlation coefficient, its p-value, 
    and a bootstrap confidence interval.

    Args:
        x (array-like): First variable.
        y (array-like): Second variable.
        n_boot (int): Number of bootstrap iterations.
        alpha (float): Significance level for the confidence interval.
        seed (int): Random seed for reproducibility.

    Returns:
        dict: Results containing the correlation coefficient (rho), 
              the p-value, and the lower and upper bounds of the CI.
    """
    rng = np.random.default_rng(random_state)
    n = len(x)
    boot_corrs = []

    x = np.asarray(x)
    y = np.asarray(y)

    # Calculate original Spearman rho and p-value
    rho, p_value = stats.spearmanr(x, y)

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_corrs.append(stats.spearmanr(x[idx], y[idx]).correlation)

    lower = np.percentile(boot_corrs, 100 * (alpha / 2))
    upper = np.percentile(boot_corrs, 100 * (1 - alpha / 2))

    return {
        "test": "spearman_correlation",
        "rho": rho,
        "p_value": p_value,
        "rho_ci": (lower, upper)
    }

## Lists

In [5]:
paired_t_tests = (
    ('Patients', 'PatScoreArzt', 'PatScoreKI'),
    ('Expert 1', 'Exp1ScoreArzt', 'Exp1ScoreKI'),
    ('Expert 2', 'Exp2ScoreArzt', 'Exp2ScoreKI')
)

wilcoxon_tests = (
    ('Patients', 'Clarity', 'PatItem1Arzt', 'PatItem1KI'),
    ('', 'Appropriateness of the content', 'PatItem2Arzt', 'PatItem2KI'),
    ('', 'Positivity of the content', 'PatItem3Arzt', 'PatItem3KI'),
    ('', 'Appropriateness of the language', 'PatItem4Arzt', 'PatItem4KI'),
    ('Expert 1', 'Clarity', 'ExpItem1Arzt', 'ExpItem1KI'),
    ('', 'Completness', 'ExpItem2Arzt', 'ExpItem2KI'),
    ('', 'Correctness', 'ExpItem3Arzt', 'ExpItem3KI'),
    ('', 'Appropriateness of the language', 'ExpItem4Arzt', 'ExpItem4KI'),
    ('Expert 2', 'Clarity', 'ExpItem1Arzt.1', 'ExpItem1KI.1'),
    ('', 'Completness', 'ExpItem2Arzt.1', 'ExpItem2KI.1'),
    ('', 'Correctness', 'ExpItem3Arzt.1', 'ExpItem3KI.1'),
    ('', 'Appropriateness of the language', 'ExpItem4Arzt.1', 'ExpItem4KI.1')
)

## Per-simplifier analyses
Each report was simplified by two physician experts, Thomas and Filippo. In the following sections, the analyses performed in `02_analysis.ipynb` are repeated separately for each simplifier.

### Main Result

In [6]:
human1 = data.loc[data['Experte']=='Thomas']
human2 = data.loc[data['Experte']=='Filippo']

#### Simplifier 1

In [7]:
human1_result = pd.DataFrame(
    columns=[
        'Rater',
        'Mean ± SD (human)',
        'Mean ± SD (AI)',
        "Cohen's d",
        "95% CI",
        'p-value'
    ]
)

for rater, human, ai in paired_t_tests:
    results = paired_t_test(human1[human], human1[ai])

    mean_sd_human = (
        f"{results['mean_array1']:.3f} ± "
        f"{results['std_array1']:.3f}"
    )

    mean_sd_ai = (
        f"{results['mean_array2']:.3f} ± "
        f"{results['std_array2']:.3f}"
    )

    ci = results.get('cohen_d_ci', (np.nan, np.nan))
    ci_str = f"[{ci[0]:.3f}, {ci[1]:.3f}]"

    row = {
        'Rater': rater,
        'Mean ± SD (human)': mean_sd_human,
        'Mean ± SD (AI)': mean_sd_ai,
        "Cohen's d": f"{results['cohen_d']:.3f}",
        "95% CI": ci_str,
        'p-value': f"{results['p_value']:.3f}"
    }

    human1_result = pd.concat(
        [human1_result, pd.DataFrame([row])],
        ignore_index=True
    )

human1_result.to_csv('./human1_result.csv', index=False)
human1_result

,Rater,Mean ± SD (human),Mean ± SD (AI),Cohen's d,95% CI,p-value
0,Patients,16.094 ± 3.888,17.688 ± 2.533,-0.409,"[-0.701, -0.089]",0.027
1,Expert 1,16.625 ± 2.152,18.188 ± 1.306,-0.592,"[-0.916, -0.309]",0.002
2,Expert 2,14.281 ± 3.008,13.875 ± 2.744,0.104,"[-0.247, 0.494]",0.559


#### Simplifier 2

In [8]:
human2_result = pd.DataFrame(
    columns=[
        'Rater',
        'Mean ± SD (human)',
        'Mean ± SD (AI)',
        "Cohen's d",
        "95% CI",
        'p-value'
    ]
)

for rater, human, ai in paired_t_tests:
    results = paired_t_test(human2[human], human2[ai])

    mean_sd_human = (
        f"{results['mean_array1']:.3f} ± "
        f"{results['std_array1']:.3f}"
    )

    mean_sd_ai = (
        f"{results['mean_array2']:.3f} ± "
        f"{results['std_array2']:.3f}"
    )

    ci = results.get('cohen_d_ci', (np.nan, np.nan))
    ci_str = f"[{ci[0]:.3f}, {ci[1]:.3f}]"

    row = {
        'Rater': rater,
        'Mean ± SD (human)': mean_sd_human,
        'Mean ± SD (AI)': mean_sd_ai,
        "Cohen's d": f"{results['cohen_d']:.3f}",
        "95% CI": ci_str,
        'p-value': f"{results['p_value']:.3f}"
    }

    human2_result = pd.concat(
        [human2_result, pd.DataFrame([row])],
        ignore_index=True
    )

human2_result.to_csv('./human2_result.csv', index=False)
human2_result

,Rater,Mean ± SD (human),Mean ± SD (AI),Cohen's d,95% CI,p-value
0,Patients,13.812 ± 4.395,16.844 ± 3.611,-0.542,"[-0.967, -0.169]",0.004
1,Expert 1,16.906 ± 1.692,17.594 ± 1.478,-0.295,"[-0.697, 0.029]",0.106
2,Expert 2,14.750 ± 1.814,13.969 ± 1.892,0.325,"[-0.015, 0.728]",0.076


#### Between simplifiers

In [9]:
between_result = pd.DataFrame(
    columns=[
        'Rater',
        'Mean ± SD (human1)',
        'Mean ± SD (human2)',
        "Cohen's d",
        "95% CI",
        'p-value'
    ]
)

for rater, human, ai in paired_t_tests:
    results = independent_t_test(human1[human], human2[human])

    mean_sd_human1 = (
        f"{results['mean_array1']:.3f} ± "
        f"{results['std_array1']:.3f}"
    )

    mean_sd_human2 = (
        f"{results['mean_array2']:.3f} ± "
        f"{results['std_array2']:.3f}"
    )

    ci = results.get('cohen_d_ci', (np.nan, np.nan))
    ci_str = f"[{ci[0]:.3f}, {ci[1]:.3f}]"

    row = {
        'Rater': rater,
        'Mean ± SD (human1)': mean_sd_human1,
        'Mean ± SD (human2)': mean_sd_human2,
        "Cohen's d": f"{results['cohen_d']:.3f}",
        "95% CI": ci_str,
        'p-value': f"{results['p_value']:.3f}"
    }

    between_result = pd.concat(
        [between_result, pd.DataFrame([row])],
        ignore_index=True
    )

between_result.to_csv('./between_result.csv', index=False)
between_result

,Rater,Mean ± SD (human1),Mean ± SD (human2),Cohen's d,95% CI,p-value
0,Patients,16.094 ± 3.888,13.812 ± 4.395,0.550,"[-1.088, -0.078]",0.032
1,Expert 1,16.625 ± 2.152,16.906 ± 1.692,-0.145,"[-0.366, 0.620]",0.563
2,Expert 2,14.281 ± 3.008,14.750 ± 1.814,-0.189,"[-0.346, 0.668]",0.453


### Semantic Similarity

In [10]:
human = 'cosine_similarity_original_vs_human'
ai = 'cosine_similarity_original_vs_AI'

#### Simplifier 1

In [11]:
paired_t_test(human1[human], human1[ai])

{'test': 'paired_t_test',
 'mean_array1': np.float64(0.7518812883645296),
 'mean_array2': np.float64(0.748625822365284),
 'std_array1': np.float64(0.03395287592927571),
 'std_array2': np.float64(0.04485570887248439),
 'cohen_d': np.float64(0.07028002932364921),
 'cohen_d_ci': (np.float64(-0.3128946909810119),
  np.float64(0.42806807879664116)),
 'statistic': np.float64(0.39756388253393415),
 'p_value': np.float64(0.6936761115398364)}

#### Simplifier 2

In [12]:
paired_t_test(human2[human], human2[ai])

{'test': 'paired_t_test',
 'mean_array1': np.float64(0.7819655053317547),
 'mean_array2': np.float64(0.7499556224793196),
 'std_array1': np.float64(0.0428722199695768),
 'std_array2': np.float64(0.03863333824449453),
 'cohen_d': np.float64(0.7475619192998737),
 'cohen_d_ci': (np.float64(0.5249770791117082),
  np.float64(1.0738495755860726)),
 'statistic': np.float64(4.228848819950171),
 'p_value': np.float64(0.00019263655902323495)}

#### Between simplifiers

In [13]:
independent_t_test(human1[human], human2[human])

{'test': 'independent_t_test',
 'mean_array1': np.float64(0.7518812883645296),
 'mean_array2': np.float64(0.7819655053317547),
 'std_array1': np.float64(0.03395287592927571),
 'std_array2': np.float64(0.0428722199695768),
 'cohen_d': np.float64(-0.7779616822602256),
 'cohen_d_ci': (np.float64(0.26640657566442866),
  np.float64(1.3848421845551806)),
 'statistic': np.float64(-3.111846729040902),
 'p_value': np.float64(0.002810606829694948)}

### Individual Items

#### Simplifier 1

In [14]:
human1_individual_items = pd.DataFrame(
    columns=[
        'Rater',
        'Item',
        'Median (IQR) human',
        'Median (IQR) AI',
        'Rank-biserial r',
        '95% CI',
        'p-value'
    ]
)

for rater, item, human, ai in wilcoxon_tests:
    results = wilcoxon_test(human1[human], human1[ai])

    # Format median and IQR for Human
    med1 = int(results['median_array1'])
    iqr1_low, iqr1_high = [float(x) for x in results['iqr_array1']]
    median_iqr_human = f"{med1} ({iqr1_low}, {iqr1_high})"

    # Format median and IQR for AI
    med2 = int(results['median_array2'])
    iqr2_low, iqr2_high = [float(x) for x in results['iqr_array2']]
    median_iqr_ai = f"{med2} ({iqr2_low}, {iqr2_high})"

    ci = results.get('rank_biserial_ci', (np.nan, np.nan))
    ci_str = f"[{ci[0]:.3f}, {ci[1]:.3f}]"

    row = {
        'Rater': rater,
        'Item': item,
        'Median (IQR) human': median_iqr_human,
        'Median (IQR) AI': median_iqr_ai,
        'Rank-biserial r': f"{results['rank_biserial']:.3f}",
        '95% CI': ci_str,
        'p-value': f"{results['p_value']:.3f}"
    }

    human1_individual_items = pd.concat(
        [human1_individual_items, pd.DataFrame([row])],
        ignore_index=True
    )

human1_individual_items.to_csv('./human1_individual_items.csv', index=False)
human1_individual_items

,Rater,Item,Median (IQR) human,Median (IQR) AI,Rank-biserial r,95% CI,p-value
0,Patients,Clarity,"5 (3.75, 5.0)","5 (4.0, 5.0)",-0.604,"[-0.927, -0.033]",0.048
1,,Appropriateness of the content,"4 (3.0, 5.0)","4 (4.0, 5.0)",-0.727,"[-1.000, -0.181]",0.030
2,,Positivity of the content,"4 (3.0, 5.0)","5 (4.0, 5.0)",-0.463,"[-0.870, 0.091]",0.096
3,,Appropriateness of the language,"4 (3.0, 5.0)","5 (4.0, 5.0)",-0.426,"[-0.831, 0.134]",0.123
4,Expert 1,Clarity,"5 (4.0, 5.0)","5 (5.0, 5.0)",-1.000,"[-1.000, -1.000]",0.003
5,,Completness,"4 (4.0, 5.0)","4 (4.0, 5.0)",0.190,"[-0.300, 0.621]",0.435
6,,Correctness,"4 (3.0, 4.0)","4 (4.0, 5.0)",-0.663,"[-0.926, -0.263]",0.007
7,,Appropriateness of the language,"4 (3.75, 5.0)","5 (4.0, 5.0)",-0.805,"[-1.000, -0.508]",0.001
8,Expert 2,Clarity,"4 (3.75, 4.0)","4 (3.0, 4.0)",-0.029,"[-0.572, 0.564]",0.922
9,,Completness,"3 (3.0, 4.0)","3 (2.0, 4.0)",0.497,"[0.105, 0.826]",0.028


#### Simplifier 2

In [15]:
human2_individual_items = pd.DataFrame(
    columns=[
        'Rater',
        'Item',
        'Median (IQR) human',
        'Median (IQR) AI',
        'Rank-biserial r',
        '95% CI',
        'p-value'
    ]
)

for rater, item, human, ai in wilcoxon_tests:
    results = wilcoxon_test(human2[human], human2[ai])

    # Format median and IQR for Human
    med1 = int(results['median_array1'])
    iqr1_low, iqr1_high = [float(x) for x in results['iqr_array1']]
    median_iqr_human = f"{med1} ({iqr1_low}, {iqr1_high})"

    # Format median and IQR for AI
    med2 = int(results['median_array2'])
    iqr2_low, iqr2_high = [float(x) for x in results['iqr_array2']]
    median_iqr_ai = f"{med2} ({iqr2_low}, {iqr2_high})"

    ci = results.get('rank_biserial_ci', (np.nan, np.nan))
    ci_str = f"[{ci[0]:.3f}, {ci[1]:.3f}]"

    row = {
        'Rater': rater,
        'Item': item,
        'Median (IQR) human': median_iqr_human,
        'Median (IQR) AI': median_iqr_ai,
        'Rank-biserial r': f"{results['rank_biserial']:.3f}",
        '95% CI': ci_str,
        'p-value': f"{results['p_value']:.3f}"
    }

    human2_individual_items = pd.concat(
        [human2_individual_items, pd.DataFrame([row])],
        ignore_index=True
    )

human2_individual_items.to_csv('./human2_individual_items.csv', index=False)
human2_individual_items

,Rater,Item,Median (IQR) human,Median (IQR) AI,Rank-biserial r,95% CI,p-value
0,Patients,Clarity,"3 (3.0, 5.0)","5 (4.0, 5.0)",-0.637,"[-1.000, -0.178]",0.013
1,,Appropriateness of the content,"3 (3.0, 4.25)","5 (3.0, 5.0)",-0.621,"[-0.909, -0.216]",0.010
2,,Positivity of the content,"3 (3.0, 5.0)","5 (3.0, 5.0)",-0.463,"[-0.829, 0.027]",0.060
3,,Appropriateness of the language,"3 (2.75, 4.0)","4 (3.0, 5.0)",-0.668,"[-0.947, -0.260]",0.005
4,Expert 1,Clarity,"4 (4.0, 4.0)","5 (4.0, 5.0)",-0.853,"[-1.000, -0.600]",0.000
5,,Completness,"5 (4.0, 5.0)","4 (4.0, 4.0)",0.917,"[0.714, 1.000]",0.000
6,,Correctness,"4 (4.0, 5.0)","4 (4.0, 5.0)",-0.200,"[-0.684, 0.228]",0.394
7,,Appropriateness of the language,"4 (4.0, 4.0)","5 (4.0, 5.0)",-0.836,"[-1.000, -0.532]",0.001
8,Expert 2,Clarity,"3 (3.0, 4.0)","4 (4.0, 4.0)",-0.581,"[-0.924, -0.187]",0.009
9,,Completness,"4 (3.0, 4.0)","3 (2.0, 4.0)",0.942,"[0.779, 1.000]",0.000


#### Between simplifiers

In [16]:
between_individual_items = pd.DataFrame(
    columns=[
        'Rater',
        'Item',
        'Median (IQR) human1',
        'Median (IQR) human2',
        'Rank-biserial r',
        '95% CI',
        'p-value'
    ]
)

for rater, item, human, ai in wilcoxon_tests:
    results = mann_whitney_u_test(human1[human], human2[human])

    # Format median and IQR for human1
    med1 = int(results['median_array1'])
    iqr1_low, iqr1_high = [float(x) for x in results['iqr_array1']]
    median_iqr_human1 = f"{med1} ({iqr1_low}, {iqr1_high})"

    # Format median and IQR for human2
    med2 = int(results['median_array2'])
    iqr2_low, iqr2_high = [float(x) for x in results['iqr_array2']]
    median_iqr_human2 = f"{med2} ({iqr2_low}, {iqr2_high})"

    ci = results.get('rank_biserial_ci', (np.nan, np.nan))
    ci_str = f"[{ci[0]:.3f}, {ci[1]:.3f}]"

    row = {
        'Rater': rater,
        'Item': item,
        'Median (IQR) human1': median_iqr_human1,
        'Median (IQR) human2': median_iqr_human2,
        'Rank-biserial r': f"{results['rank_biserial']:.3f}",
        '95% CI': ci_str,
        'p-value': f"{results['p_value']:.3f}"
    }

    between_individual_items = pd.concat(
        [between_individual_items, pd.DataFrame([row])],
        ignore_index=True
    )

between_individual_items.to_csv('./between_individual_items.csv', index=False)
between_individual_items

,Rater,Item,Median (IQR) human1,Median (IQR) human2,Rank-biserial r,95% CI,p-value
0,Patients,Clarity,"5 (3.75, 5.0)","3 (3.0, 5.0)",0.315,"[0.050, 0.557]",0.021
1,,Appropriateness of the content,"4 (3.0, 5.0)","3 (3.0, 4.25)",0.213,"[-0.069, 0.484]",0.131
2,,Positivity of the content,"4 (3.0, 5.0)","3 (3.0, 5.0)",0.197,"[-0.068, 0.478]",0.162
3,,Appropriateness of the language,"4 (3.0, 5.0)","3 (2.75, 4.0)",0.328,"[0.045, 0.578]",0.020
4,Expert 1,Clarity,"5 (4.0, 5.0)","4 (4.0, 4.0)",0.352,"[0.103, 0.580]",0.008
5,,Completness,"4 (4.0, 5.0)","5 (4.0, 5.0)",-0.201,"[-0.434, 0.047]",0.112
6,,Correctness,"4 (3.0, 4.0)","4 (4.0, 5.0)",-0.251,"[-0.481, -0.003]",0.051
7,,Appropriateness of the language,"4 (3.75, 5.0)","4 (4.0, 4.0)",-0.002,"[-0.246, 0.258]",0.994
8,Expert 2,Clarity,"4 (3.75, 4.0)","3 (3.0, 4.0)",0.265,"[-0.006, 0.507]",0.046
9,,Completness,"3 (3.0, 4.0)","4 (3.0, 4.0)",-0.294,"[-0.534, -0.037]",0.029


## Text Length

### Correlation

#### AI simplifications

In [17]:
results = spearman_rho(data['PatScoreKI'].values, data['AI_translation_length'].values)

# Print results
print(f"{'Spearman correlation:':<25} {results['rho']:>10.3f}")
print(f"{'95% CI:':<25}      [{results['rho_ci'][0]:.3f}, {results['rho_ci'][1]:.3f}]")
print(f"{'p-value:':<25} {results['p_value']:>10.3f}")

Spearman correlation:         -0.151
95% CI:                        [-0.399, 0.114]
p-value:                       0.233


#### Human simplifications

In [18]:
results = spearman_rho(data['PatScoreArzt'].values, data['human_translation_length'].values)

# Print results
print(f"{'Spearman correlation:':<25} {results['rho']:>10.3f}")
print(f"{'95% CI:':<25}      [{results['rho_ci'][0]:.3f}, {results['rho_ci'][1]:.3f}]")
print(f"{'p-value:':<25} {results['p_value']:>10.3f}")

Spearman correlation:         -0.084
95% CI:                        [-0.302, 0.153]
p-value:                       0.511


### Oridnal Regression

#### Rater 1

In [19]:
df_human = pd.DataFrame({
    'Completeness': data['ExpItem2Arzt'].astype(int),
    'Length': data['human_translation_length'].astype(int),
    'Type': 'Human'
})

df_ai = pd.DataFrame({
    'Completeness': data['ExpItem2KI'].astype(int),
    'Length': data['AI_translation_length'].astype(int),
    'Type': 'AI'
})

df_long = pd.concat([df_human, df_ai], ignore_index=True)

# Collapse low-frequency categories
def collapse_completeness(val):
    if val <= 2:
        return 1
    return val - 1

df_long['Completeness_collapsed'] = df_long['Completeness'].apply(collapse_completeness)

df_long['Completeness_collapsed'] = pd.Categorical(
    df_long['Completeness_collapsed'],
    categories=[1, 2, 3, 4],
    ordered=True
)

# Proper binary encoding (Human=0, AI=1)
df_long['Type'] = (df_long['Type'] == 'AI').astype(int)

# Fit ordinal logistic regression
model = OrderedModel.from_formula(
    "Completeness_collapsed ~ Type * Length",
    data=df_long,
    distr='logit'
)

result = model.fit(method='bfgs')
result.summary()

Optimization terminated successfully.
         Current function value: 0.917146
         Iterations: 34
         Function evaluations: 39
         Gradient evaluations: 39


<class 'statsmodels.iolib.summary.Summary'>
"""
                               OrderedModel Results                               
==================================================================================
Dep. Variable:     Completeness_collapsed   Log-Likelihood:                -117.39
Model:                       OrderedModel   AIC:                             246.8
Method:                Maximum Likelihood   BIC:                             263.9
Date:                    Tue, 30 Jun 2026                                         
Time:                            16:48:19                                         
No. Observations:                     128                                         
Df Residuals:                         122                                         
Df Model:                               3                                         
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Type           -1.7903      2.402     -0.745      0.456      -6.498       2.918
Length          0.0004      0.000      1.538      0.124      -0.000       0.001
Type:Length     0.0006      0.002      0.386      0.700      -0.002       0.004
1/2            -4.7336      1.190     -3.977      0.000      -7.066      -2.401
2/3             0.9919      0.360      2.753      0.006       0.286       1.698
3/4             0.9668      0.120      8.032      0.000       0.731       1.203
===============================================================================
"""

In [20]:
scale = 100  # 100-character increase

params = result.params
conf = result.conf_int()

# copy so we don't modify unrelated terms
params_adj = params.copy()
conf_adj = conf.copy()

# scale ONLY length-related terms (effect of +100 characters)
params_adj["Length"] = params["Length"] * scale
conf_adj.loc["Length"] = conf.loc["Length"] * scale

params_adj["Type:Length"] = params["Type:Length"] * scale
conf_adj.loc["Type:Length"] = conf.loc["Type:Length"] * scale

# convert to OR
or_100 = np.exp(params_adj)
ci_100 = np.exp(conf_adj)

effect_sizes_100 = pd.DataFrame({
    "Odds Ratio": or_100,
    "CI 2.5%": ci_100[0],
    "CI 97.5%": ci_100[1]
})

effect_sizes_100

,Odds Ratio,CI 2.5%,CI 97.5%
Type,0.166907,0.001506,18.495996
Length,1.038976,0.989559,1.090861
Type:Length,1.060479,0.786968,1.429050
1/2,0.008795,0.000853,0.090647
2/3,2.696223,1.330584,5.463477
3/4,2.629468,2.076855,3.329121


#### Rater 2

In [21]:
df_human = pd.DataFrame({
    'Completeness': data['ExpItem2Arzt.1'].astype(int),
    'Length': data['human_translation_length'].astype(int),
    'Type': 'Human'
})

df_ai = pd.DataFrame({
    'Completeness': data['ExpItem2KI.1'].astype(int),
    'Length': data['AI_translation_length'].astype(int),
    'Type': 'AI'
})

df_long = pd.concat([df_human, df_ai], ignore_index=True)

# Collapse low-frequency categories
def collapse_completeness(val):
    if val <= 2:
        return 1
    return val - 1

df_long['Completeness_collapsed'] = df_long['Completeness'].apply(collapse_completeness)

df_long['Completeness_collapsed'] = pd.Categorical(
    df_long['Completeness_collapsed'],
    categories=[1, 2, 3, 4],
    ordered=True
)

# Proper binary encoding (Human=0, AI=1)
df_long['Type'] = (df_long['Type'] == 'AI').astype(int)

# Fit ordinal logistic regression
model = OrderedModel.from_formula(
    "Completeness_collapsed ~ Type * Length",
    data=df_long,
    distr='logit'
)

result = model.fit(method='bfgs')
result.summary()

Optimization terminated successfully.
         Current function value: 1.179144
         Iterations: 29
         Function evaluations: 34
         Gradient evaluations: 34


<class 'statsmodels.iolib.summary.Summary'>
"""
                               OrderedModel Results                               
==================================================================================
Dep. Variable:     Completeness_collapsed   Log-Likelihood:                -150.93
Model:                       OrderedModel   AIC:                             313.9
Method:                Maximum Likelihood   BIC:                             331.0
Date:                    Tue, 30 Jun 2026                                         
Time:                            16:48:19                                         
No. Observations:                     128                                         
Df Residuals:                         122                                         
Df Model:                               3                                         
===============================================================================
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Type           -0.2545      2.257     -0.113      0.910      -4.679       4.170
Length          0.0004      0.000      1.532      0.126   -9.86e-05       0.001
Type:Length    -0.0006      0.001     -0.387      0.699      -0.003       0.002
1/2            -1.1130      0.617     -1.804      0.071      -2.322       0.096
2/3             0.3747      0.147      2.548      0.011       0.086       0.663
3/4             0.9724      0.138      7.050      0.000       0.702       1.243
===============================================================================
"""

In [22]:
scale = 100  # 100-character increase

params = result.params
conf = result.conf_int()

# copy so we don't modify unrelated terms
params_adj = params.copy()
conf_adj = conf.copy()

# scale ONLY length-related terms (effect of +100 characters)
params_adj["Length"] = params["Length"] * scale
conf_adj.loc["Length"] = conf.loc["Length"] * scale

params_adj["Type:Length"] = params["Type:Length"] * scale
conf_adj.loc["Type:Length"] = conf.loc["Type:Length"] * scale

# convert to OR
or_100 = np.exp(params_adj)
ci_100 = np.exp(conf_adj)

effect_sizes_100 = pd.DataFrame({
    "Odds Ratio": or_100,
    "CI 2.5%": ci_100[0],
    "CI 97.5%": ci_100[1]
})

effect_sizes_100

,Odds Ratio,CI 2.5%,CI 97.5%
Type,0.775285,0.009288,64.717577
Length,1.035930,0.990185,1.083789
Type:Length,0.945439,0.711744,1.255866
1/2,0.328572,0.098033,1.101258
2/3,1.454514,1.090329,1.940340
3/4,2.644249,2.017862,3.465081


## Inter-rater Reliability

In [23]:
exp1 = pd.concat([
    data['Exp1ScoreArzt'],
    data['Exp1ScoreKI']
], ignore_index=True).values

exp2 = pd.concat([
    data['Exp2ScoreArzt'],
    data['Exp2ScoreKI']
], ignore_index=True).values

In [24]:
ratings = np.vstack([exp1, exp2]).T
n, k = ratings.shape

mean_per_target = np.mean(ratings, axis=1)
mean_per_rater = np.mean(ratings, axis=0)
grand_mean = np.mean(ratings)

SS_between = k * np.sum((mean_per_target - grand_mean) ** 2)
SS_within = np.sum((ratings - mean_per_target[:, None]) ** 2)

MS_between = SS_between / (n - 1)
MS_within = SS_within / (n * (k - 1))

icc = (MS_between - MS_within) / (MS_between + (k - 1) * MS_within)
icc

np.float64(-0.33906015842432025)